# MSME-RL — Trained Agent Demo

**What this notebook shows:** the same Indian-banking environment, played twice on the same fixed seed:

1. **Random baseline** — the obvious null hypothesis: a policy that picks actions uniformly.
2. **Trained policy** — Qwen2.5-1.5B-Instruct after SFT warm-start + GRPO with KL anchor and entropy bonus.

Two case studies are highlighted:

- An **MSME borrower** who *understates* stress in Hindi/Hinglish.
- A **startup founder** who *overstates* health in English.

For each, you'll see the agent's chosen action and the resulting step reward. The aggregate comparison at the bottom is the hard-number version of the same claim.

> All trajectories are **reproducible from this notebook**: same env seed, same model checkpoint, same tokenizer settings. Run top-to-bottom on a CUDA box with the trained checkpoint already on disk.

## 1. Setup

Configuration is in one place at the top of the next cell. Edit `TRAINED_CKPT` to point at the checkpoint you want to evaluate (e.g. `episode_0006`, `episode_0010`, ...).

In [ ]:
import os, sys, json, random, gc
from typing import Any, Dict, List

# ---- Edit these ---------------------------------------------------------- #
BASE_MODEL    = "Qwen/Qwen2.5-1.5B-Instruct"
TRAINED_CKPT  = "msme_rl_checkpoints_risk30_q25/episode_0006"
SEED          = 42
EPISODES      = 3
MAX_STEPS     = 60                        # one episode = 36-month portfolio loop
ARTIFACTS_DIR = "artifacts"

os.makedirs(ARTIFACTS_DIR, exist_ok=True)
sys.path.insert(0, os.path.abspath("."))  # so `import train_grpo` works

import torch
print(f"torch={torch.__version__}  cuda={torch.cuda.is_available()}  device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"trained checkpoint exists: {os.path.isdir(TRAINED_CKPT)} ({TRAINED_CKPT})")


## 2. Environment + helper functions

Reuses `MSMERLEnvironment`, `MSMERLAction`, `build_agent_prompt`, `_parse_action_from_text`, `_snap_to_valid_action`, and the `JSON_PREFILL` constant from `train_grpo.py` so the inference path is byte-identical to training-time rollouts.

`run_episode(...)` is a small driver that produces a list of `(observation, action, reward, info)` records — one per step — for any callable `policy(obs) -> MSMERLAction`.

In [ ]:
from server.msmeEnv_environment import MSMERLEnvironment
from models import MSMERLAction
from train_grpo import (
    VALID_ACTIONS,
    SYSTEM_PROMPT,
    build_agent_prompt,
)

def _seed_everything(seed: int) -> None:
    random.seed(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except Exception:
        pass
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def run_episode(policy, seed: int, max_steps: int = MAX_STEPS) -> Dict[str, Any]:
    """Run a single episode against MSMERLEnvironment with a callable policy.

    `policy` is any function taking an observation dict and returning an
    `MSMERLAction`. Used both for the random baseline and the trained model.
    """
    _seed_everything(seed)
    env = MSMERLEnvironment()
    obs_obj = env.reset()
    obs = obs_obj.__dict__ if hasattr(obs_obj, "__dict__") else dict(obs_obj)

    trace = []
    cumulative_reward = 0.0
    for step_idx in range(max_steps):
        action = policy(obs)
        out = env.step(action)
        new_obs = out.__dict__ if hasattr(out, "__dict__") else dict(out)
        step_reward = new_obs.get("step_reward", 0.0)
        cumulative_reward += step_reward
        trace.append({
            "step":        step_idx,
            "action_type": action.action_type,
            "account_id":  action.account_id,
            "reasoning":   getattr(action, "reasoning", "") or "",
            "step_reward": step_reward,
            "cum_reward":  cumulative_reward,
            "month":       (new_obs.get("portfolio_summary", {}) or {}).get("current_month"),
            "npa_rate":    (new_obs.get("portfolio_summary", {}) or {}).get("npa_rate"),
            "trust":       (new_obs.get("portfolio_summary", {}) or {}).get("avg_trust_score"),
        })
        obs = new_obs
        if new_obs.get("done", False):
            break

    summary = obs.get("portfolio_summary", {}) or {}
    return {
        "trace":           trace,
        "total_reward":    cumulative_reward,
        "final_npa_rate":  summary.get("npa_rate"),
        "final_trust":     summary.get("avg_trust_score"),
        "steps_taken":     len(trace),
    }


print("Helpers ready. VALID_ACTIONS:", len(VALID_ACTIONS), "options")


## 3. Random baseline

This is the null hypothesis: a policy that picks any of the 21 valid action types uniformly at random against any of the 30 accounts. If the trained policy can't beat this, training did nothing.

In [ ]:
def random_policy(obs: Dict[str, Any]) -> MSMERLAction:
    return MSMERLAction(
        action_type=random.choice(VALID_ACTIONS),
        account_id=random.randint(1, 30),
        parameters={},
        reasoning="random baseline",
    )

random_runs = []
for ep in range(EPISODES):
    seed_ep = SEED + ep
    result  = run_episode(random_policy, seed=seed_ep)
    result["seed"]    = seed_ep
    result["episode"] = ep + 1
    random_runs.append(result)
    print(
        f"  Random ep {ep+1} (seed={seed_ep}): reward={result['total_reward']:+.3f} | "
        f"NPA={result['final_npa_rate']:.1%} | trust={result['final_trust']:.2f}"
    )

random_mean = sum(r["total_reward"] for r in random_runs) / len(random_runs)
print(f"\nRandom baseline mean reward over {EPISODES} episodes: {random_mean:+.3f}")


## 4. Trained policy

Loads the trained checkpoint (`TRAINED_CKPT`) and wraps it in a `trained_policy` callable with the same interface as the random one. Uses the **identical** prompt builder and JSON-prefill / parser path as `train_grpo.py` so what you see here is what the policy actually does in production.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import re, ast

JSON_PREFILL = '{"action_type": "'

print(f"Loading tokenizer from {TRAINED_CKPT} ...")
tok_path = TRAINED_CKPT if os.path.isfile(os.path.join(TRAINED_CKPT, "tokenizer_config.json")) else BASE_MODEL
tokenizer = AutoTokenizer.from_pretrained(tok_path, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Loading model from {TRAINED_CKPT} ...")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.bfloat16 if (device == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    TRAINED_CKPT,
    torch_dtype=dtype,
    device_map=device,
    trust_remote_code=True,
)
model.eval()
print(f"Model loaded ({dtype}, {sum(p.numel() for p in model.parameters())/1e9:.2f}B params)")


In [ ]:
def _build_full_prompt(user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


def _parse_action(generated: str, obs: Dict[str, Any]) -> MSMERLAction:
    from train_grpo import _snap_to_valid_action
    clean = generated.strip()
    if clean.startswith("```"):
        parts = clean.split("```")
        clean = parts[1] if len(parts) > 1 else clean
        if clean.startswith("json"):
            clean = clean[4:]
    clean = clean.strip()
    candidates = [clean]
    for span in re.findall(r"\{.*?\}", clean, flags=re.DOTALL):
        if span not in candidates:
            candidates.append(span)
    li, ri = clean.find("{"), clean.rfind("}")
    if li != -1 and ri != -1 and ri > li:
        candidates.append(clean[li:ri+1])

    def _to_action(d: Dict[str, Any]) -> MSMERLAction:
        try:
            acc = int(str(d.get("account_id", 1)).strip())
        except Exception:
            acc = 1
        params = d.get("parameters", {})
        if not isinstance(params, dict):
            params = {}
        return MSMERLAction(
            action_type=_snap_to_valid_action(str(d.get("action_type", "wait_and_observe"))),
            account_id=max(1, min(30, acc)),
            parameters=params,
            reasoning=str(d.get("reasoning", "")),
        )

    for c in candidates:
        for loader in (json.loads, ast.literal_eval):
            try:
                d = loader(c)
                if isinstance(d, dict):
                    return _to_action(d)
            except Exception:
                pass
    am = re.search(r"(action_type|action)\s*[:=]\s*['\"]?([a-zA-Z0-9_]+)['\"]?", clean)
    aid = re.search(r"(account_id|account)\s*[:=]\s*([0-9]+)", clean)
    if am:
        return MSMERLAction(
            action_type=_snap_to_valid_action(am.group(2)),
            account_id=max(1, min(30, int(aid.group(2)) if aid else 1)),
            parameters={},
            reasoning="(regex-recovered)",
        )
    return MSMERLAction(action_type="wait_and_observe", account_id=1, parameters={}, reasoning="(unparseable)")


@torch.inference_mode()
def trained_policy(obs: Dict[str, Any]) -> MSMERLAction:
    prompt          = build_agent_prompt(obs)
    full_prompt     = _build_full_prompt(prompt) + JSON_PREFILL
    inputs          = tokenizer(full_prompt, return_tensors="pt").to(device)
    generated_ids   = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,                 # deterministic for the demo
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.15,
        no_repeat_ngram_size=4,
    )
    suffix          = tokenizer.decode(generated_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    full_generation = JSON_PREFILL + suffix
    return _parse_action(full_generation, obs)


In [ ]:
trained_runs = []
for ep in range(EPISODES):
    seed_ep = SEED + ep
    result  = run_episode(trained_policy, seed=seed_ep)
    result["seed"]    = seed_ep
    result["episode"] = ep + 1
    trained_runs.append(result)
    print(
        f"  Trained ep {ep+1} (seed={seed_ep}): reward={result['total_reward']:+.3f} | "
        f"NPA={result['final_npa_rate']:.1%} | trust={result['final_trust']:.2f}"
    )

trained_mean = sum(r["total_reward"] for r in trained_runs) / len(trained_runs)
print(f"\nTrained policy mean reward over {EPISODES} episodes: {trained_mean:+.3f}")
print(f"Improvement vs random:                                   {trained_mean - random_mean:+.3f}")


## 5. Case study A — MSME borrower (understatement in Hindi/Hinglish)

The MSME-side accounts in this environment generate messages like *"sab theek hai sir, thoda time aur de dijiye"* even when the underlying GST returns are missed and the cluster is showing centrality stress.

The cell below picks the **first MSME-targeted action** from each policy on episode 1, so you can compare what each policy decided to do given the same step.

In [ ]:
def first_action_against(trace: List[Dict[str, Any]], account_range) -> Dict[str, Any]:
    for step in trace:
        if step["account_id"] in account_range:
            return step
    return trace[0] if trace else {}

MSME_RANGE    = range(1, 21)   # accounts 1-20 are MSME
STARTUP_RANGE = range(21, 31)  # accounts 21-30 are startup

print("=" * 72)
print("CASE A — first MSME-targeted decision in episode 1")
print("=" * 72)
ra = first_action_against(random_runs[0]["trace"], MSME_RANGE)
ta = first_action_against(trained_runs[0]["trace"], MSME_RANGE)
print(f"\n[RANDOM]  step {ra.get('step'):>2}: account={ra.get('account_id')} | "
      f"action={ra.get('action_type'):<32} step_reward={ra.get('step_reward'):+.3f}")
print(f"[TRAINED] step {ta.get('step'):>2}: account={ta.get('account_id')} | "
      f"action={ta.get('action_type'):<32} step_reward={ta.get('step_reward'):+.3f}")
if ta.get("reasoning"):
    print(f"          reasoning: {ta['reasoning'][:200]}")
print("\nWhat the trained policy is doing here:")
print("  - Picks a domain-specific intervention (vs uniform random)")
print(f"  - Step reward delta: {ta.get('step_reward', 0) - ra.get('step_reward', 0):+.3f}")


## 6. Case study B — Startup founder (overstatement in English)

Startup-side accounts (21-30) generate confident messages like *"we just closed our pre-Series B, all good"* even when investor updates have stopped 90 days ago and runway is < 6 weeks.

The danger here is *believing the pitch and waiting*. The right move is `request_investor_update_meeting` (which carries the +0.10 `investor_meeting_triggered_bridge` outcome on a genuinely stressed startup). SARFAESI on a startup is `-0.15` plus the regular NPA penalty — it should never fire here.

In [ ]:
print("=" * 72)
print("CASE B — first startup-targeted decision in episode 1")
print("=" * 72)
rb = first_action_against(random_runs[0]["trace"], STARTUP_RANGE)
tb = first_action_against(trained_runs[0]["trace"], STARTUP_RANGE)
print(f"\n[RANDOM]  step {rb.get('step'):>2}: account={rb.get('account_id')} | "
      f"action={rb.get('action_type'):<32} step_reward={rb.get('step_reward'):+.3f}")
print(f"[TRAINED] step {tb.get('step'):>2}: account={tb.get('account_id')} | "
      f"action={tb.get('action_type'):<32} step_reward={tb.get('step_reward'):+.3f}")
if tb.get("reasoning"):
    print(f"          reasoning: {tb['reasoning'][:200]}")

# Sanity check: trained policy should NEVER fire SARFAESI on a startup.
trained_sarfaesi_on_startup = sum(
    1 for r in trained_runs for s in r["trace"]
    if s["action_type"] == "initiate_sarfaesi" and s["account_id"] in STARTUP_RANGE
)
random_sarfaesi_on_startup = sum(
    1 for r in random_runs for s in r["trace"]
    if s["action_type"] == "initiate_sarfaesi" and s["account_id"] in STARTUP_RANGE
)
print(f"\nSARFAESI-on-startup count across {EPISODES} episodes:")
print(f"  RANDOM : {random_sarfaesi_on_startup}")
print(f"  TRAINED: {trained_sarfaesi_on_startup}    (lower is correct — wrong tool for startups)")


## 7. Aggregate comparison

Same seeds, same env, same step budget — only the policy changes.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

random_rewards   = [r["total_reward"] for r in random_runs]
trained_rewards  = [r["total_reward"] for r in trained_runs]

random_npa       = [r["final_npa_rate"]  or 0 for r in random_runs]
trained_npa      = [r["final_npa_rate"]  or 0 for r in trained_runs]

random_trust     = [r["final_trust"]     or 0 for r in random_runs]
trained_trust    = [r["final_trust"]     or 0 for r in trained_runs]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

x = np.arange(EPISODES)
w = 0.38

axes[0].bar(x - w/2, random_rewards,  w, label="Random",  color="#bbbbbb")
axes[0].bar(x + w/2, trained_rewards, w, label="Trained", color="#1f77b4")
axes[0].set_title("Total Episode Reward")
axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Cumulative reward")
axes[0].set_xticks(x); axes[0].set_xticklabels([f"Ep {i+1}" for i in range(EPISODES)])
axes[0].axhline(0, color="black", linewidth=0.6)
axes[0].grid(axis="y", linestyle="--", alpha=0.4)
axes[0].legend(loc="best")

axes[1].bar(x - w/2, [v*100 for v in random_npa],  w, label="Random",  color="#bbbbbb")
axes[1].bar(x + w/2, [v*100 for v in trained_npa], w, label="Trained", color="#d62728")
axes[1].set_title("Final NPA Rate (lower is better)")
axes[1].set_xlabel("Episode"); axes[1].set_ylabel("NPA %")
axes[1].set_xticks(x); axes[1].set_xticklabels([f"Ep {i+1}" for i in range(EPISODES)])
axes[1].grid(axis="y", linestyle="--", alpha=0.4)
axes[1].legend(loc="best")

axes[2].bar(x - w/2, random_trust,  w, label="Random",  color="#bbbbbb")
axes[2].bar(x + w/2, trained_trust, w, label="Trained", color="#2ca02c")
axes[2].set_title("Final Trust Score (higher is better)")
axes[2].set_xlabel("Episode"); axes[2].set_ylabel("Trust")
axes[2].set_xticks(x); axes[2].set_xticklabels([f"Ep {i+1}" for i in range(EPISODES)])
axes[2].grid(axis="y", linestyle="--", alpha=0.4)
axes[2].legend(loc="best")

fig.suptitle("MSME-RL: Random vs Trained Policy (matched seeds)", fontsize=14, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.94))

out_png = os.path.join(ARTIFACTS_DIR, "inference_comparison.png")
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_png}")


## 8. Action diversity

Random fires every action ~5% of the time. The trained policy concentrates probability mass on the actions that matter for *this* environment (information gathering + targeted intervention) and avoids the mass-coercion actions (`initiate_sarfaesi`, `file_drt_case`, `refer_to_recovery_agent`) except when warranted.

If the trained bar chart looked the same as the random one, training would have been a no-op.

In [ ]:
from collections import Counter

def action_distribution(runs):
    c = Counter()
    total = 0
    for r in runs:
        for s in r["trace"]:
            c[s["action_type"]] += 1
            total += 1
    return {a: c.get(a, 0) / max(1, total) for a in VALID_ACTIONS}

random_dist  = action_distribution(random_runs)
trained_dist = action_distribution(trained_runs)

ordered = sorted(VALID_ACTIONS, key=lambda a: -trained_dist[a])
xs = np.arange(len(ordered))
w  = 0.4

fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(xs - w/2, [random_dist[a]*100  for a in ordered], w, label="Random",  color="#bbbbbb")
ax.bar(xs + w/2, [trained_dist[a]*100 for a in ordered], w, label="Trained", color="#1f77b4")
ax.set_xticks(xs); ax.set_xticklabels(ordered, rotation=60, ha="right")
ax.set_ylabel("% of all steps"); ax.set_title("Action distribution: Random vs Trained")
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.legend(loc="best")
plt.tight_layout()
out_dist = os.path.join(ARTIFACTS_DIR, "inference_action_distribution.png")
plt.savefig(out_dist, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_dist}")


## 9. Save audit JSON

Everything above is also dumped to `artifacts/inference_comparison.json` so a judge can re-derive the numbers without rerunning the notebook.

In [ ]:
audit = {
    "config": {
        "base_model":      BASE_MODEL,
        "trained_ckpt":    TRAINED_CKPT,
        "seed":            SEED,
        "episodes":        EPISODES,
        "max_steps":       MAX_STEPS,
    },
    "random": {
        "per_episode": [{"seed": r["seed"], "reward": r["total_reward"], "npa": r["final_npa_rate"], "trust": r["final_trust"]} for r in random_runs],
        "mean_reward": random_mean,
    },
    "trained": {
        "per_episode": [{"seed": r["seed"], "reward": r["total_reward"], "npa": r["final_npa_rate"], "trust": r["final_trust"]} for r in trained_runs],
        "mean_reward": trained_mean,
    },
    "improvement_vs_random": trained_mean - random_mean,
    "sarfaesi_on_startup_count": {
        "random":  random_sarfaesi_on_startup,
        "trained": trained_sarfaesi_on_startup,
    },
    "action_distribution": {
        "random":  random_dist,
        "trained": trained_dist,
    },
}

audit_path = os.path.join(ARTIFACTS_DIR, "inference_comparison.json")
with open(audit_path, "w") as f:
    json.dump(audit, f, indent=2)

print(f"Saved: {audit_path}")
print()
print("=" * 72)
print("SUMMARY")
print("=" * 72)
print(f"Random  mean reward over {EPISODES} ep: {random_mean:+.3f}")
print(f"Trained mean reward over {EPISODES} ep: {trained_mean:+.3f}")
print(f"Improvement:                            {trained_mean - random_mean:+.3f}")
print(f"SARFAESI-on-startup (random / trained): {random_sarfaesi_on_startup} / {trained_sarfaesi_on_startup}")


## 10. (Optional) Free GPU memory

Run this when you're done with the demo so the kernel releases the trained model.

In [ ]:
try:
    del model
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("GPU memory released.")
